# LogiScan Stage 3 — v1.3 Retrain (29 classes)

Phase 8 #5/#8: retrain the Stage 3 fine classifier on `unified_training_data_v1.3.json` (17,938 samples; 16,942 labeled across 29 classes, including the rescued formal classes).

**Runtime:** GPU T4 x2 (DataParallel, batch 16/GPU) or P100 · **Expected wall time:** 30–90 min for 6 epochs on DeBERTa-v3-small.

**Targets:** macro-F1 ≥ the phase4 baseline, `affirming_consequent` recall > 0.39, all five formerly-zero formal classes with recall > 0.

In [ ]:
# 1. Install dependencies
!pip install -q transformers==4.44.2 datasets torch scikit-learn tqdm onnx

In [ ]:
# 2. Verify GPU
import torch
print(f"GPUs: {torch.cuda.device_count()} | {[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}")

In [ ]:
# 3. Upload your dataset
# Create a Kaggle dataset from the local file:
#   data/unified_training_data_v1.3.json
# Upload it (private is fine) and note the dataset slug below.

import json
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

DATA_PATH = "/kaggle/input/logiscan-v13/unified_training_data_v1.3.json"

data = json.load(open(DATA_PATH))
labeled = [d for d in data if "fallacy" in d]
print(f"Total: {len(data)} | Labeled: {len(labeled)} | Dropped near-miss: {len(data) - len(labeled)}")

texts = [d["text"] for d in labeled]
label_list = sorted(set(d["fallacy"] for d in labeled))
label2id = {l: i for i, l in enumerate(label_list)}
labels = [label2id[d["fallacy"]] for d in labeled]
print(f"Classes: {len(label_list)}")

In [ ]:
# 4. Dataset + weights
class FallacyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
counts = np.bincount(labels, minlength=len(label_list))
weights = 1.0 / (np.sqrt(counts) + 1e-6)
weights = weights / weights.sum() * len(label_list)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print(f"Imbalance ratio: {counts.max() / counts.min():.1f}:1")

X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.1, random_state=42, stratify=labels
)
print(f"Train: {len(X_train)} / Val: {len(X_val)}")

In [ ]:
# 5. Model setup (matches production: DeBERTa-v3-small, single head)
model_name = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_ds = FallacyDataset(X_train, y_train, tokenizer)
val_ds = FallacyDataset(X_val, y_val, tokenizer)
n_gpus = torch.cuda.device_count() if device.type == "cuda" else 0
loader_batch = 16 * n_gpus if n_gpus > 1 else 16
train_loader = DataLoader(train_ds, batch_size=loader_batch, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=loader_batch * 2)

config = AutoConfig.from_pretrained(model_name)
config.num_labels = len(label_list)
config.id2label = {i: l for i, l in enumerate(label_list)}
config.label2id = {l: i for i, l in enumerate(label_list)}
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, config=config, ignore_mismatched_sizes=True
).to(device)
if n_gpus > 1:
    torch.backends.cudnn.benchmark = True
    model = nn.DataParallel(model)
    print(f"DataParallel across {n_gpus} GPUs, batch {loader_batch} total ({16}/GPU)")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 6. Training loop (6 epochs, AMP, sqrt-inverse-frequency CE)
epochs = 6
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)
scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

best_f1 = 0.0
formal = [l for l in label_list if l in {
    "affirming_consequent", "denying_antecedent", "undistributed_middle",
    "illicit_major", "illicit_minor", "exclusive_premises", "existential_fallacy",
}]

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        optimizer.zero_grad()
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        y = batch["labels"].to(device)
        if scaler:
            with torch.amp.autocast(device_type="cuda"):
                loss = loss_fn(model(ids, attention_mask=mask).logits, y)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = loss_fn(model(ids, attention_mask=mask).logits, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        train_loss += loss.item()

    model.eval()
    preds_all, y_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch["input_ids"].to(device), attention_mask=batch["attention_mask"].to(device))
            preds_all.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
            y_all.extend(batch["labels"].numpy())

    f1 = f1_score(y_all, preds_all, average="macro")
    print(f"Epoch {epoch+1}: loss={train_loss/len(train_loader):.4f} val_macro_f1={f1:.4f}")
    print(classification_report(y_all, preds_all, target_names=label_list, zero_division=0, digits=3))
    for l in formal:
        i = label_list.index(l)
        r = f1_score([1 if y == i else 0 for y in y_all], [1 if p == i else 0 for p in preds_all], zero_division=0)
        print(f"  formal {l}: F1={r:.3f}")

    if f1 > best_f1:
        best_f1 = f1
        save_model = model.module if isinstance(model, nn.DataParallel) else model
        save_model.save_pretrained("stage3_v13_classifier")
        tokenizer.save_pretrained("stage3_v13_classifier")
        json.dump(label_list, open("stage3_v13_classifier/labels.json", "w"), indent=2)

print(f"\nBest macro F1: {best_f1:.4f}")

In [ ]:
# 7. Package for download (zip the model folder)
!zip -r stage3_v13_classifier.zip stage3_v13_classifier/
import os
print(f"Size: {os.path.getsize('stage3_v13_classifier.zip')/1024**2:.1f} MB")

In [ ]:
# 8. Download instructions (run in a notebook cell on the local machine):
#   !kaggle kernels output <username>/<notebook-slug> -p models/stage3_v13_classifier
# Then update STAGE3_MODEL_PATH / deployment model mounts and re-run the suite.